# Kaggriculture submission v4: fast market execution

V4 responds to the live market without destabilizing v2's productive core.
Local ablation showed that chasing short-lived spot prices with long-lag crop
changes was harmful, so this version preserves v2's 25-plot rotation and reacts
at the execution layer instead:

- tracks exact inventory movement after removing known town demand and its own
  prior sales;
- counts duplicate shop instances rather than collapsing them into a set;
- watches the opponent's visible crop maturity and recent harvest pipeline;
- assigns at most one courier to vulnerable inventory when competing premium
  supply is visible;
- can drop produce at a shed-access tile and sell it in that same turn because
  unit actions execute before market orders;
- waits through a material shop-consumption tick when there is no capacity,
  liquidation, or competing-supply reason to sell first;
- retains v2's tested batch sizes, price floors, labor, crop layout, and
  endgame liquidation.

The agent is deterministic, self-contained, and performs no network or
filesystem access during an episode.

Official references: [competition overview](https://www.kaggle.com/competitions/kaggriculture/overview), [rules](https://www.kaggle.com/competitions/kaggriculture/rules).

## 1. Install the same environment version used for local validation

This installation is only for building/testing the notebook. The submitted `main.py` itself has no external dependencies.

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments==1.32.7"

## 2. Write the required v4 entrypoint

The submitted configuration uses stable planting and fast sale execution. Its
module memory is keyed by player and automatically resets between episodes, so
the exact same file is safe for Kaggle's self-play Validation Episode.

In [ ]:
%%writefile main.py
"""Kaggriculture v4: stable production with fast market execution.

V4 preserves v2's proven 25-plot crop rotation.  It reacts where observations
are timely enough to be reliable: delivery and selling.  The agent tracks exact
market inventory, recent net supply, duplicate town-shop demand, and the visible
opponent harvest pipeline.  One courier can route vulnerable produce to the
shed for same-turn sale, while ordinary sales wait through a known town-demand
tick when capacity and competing supply permit.
"""

import math


PASS = ["PASS"]
DESIRED_HANDS = 5
MAX_MARKET_ORDERS = 10
SOFT_EXPOSURE = 55
HARD_EXPOSURE = 82
FINAL_PLANT_HOUR = 20
FINAL_FARM_DAY = 28
SOFT_LIQUIDATION_DAY = 27
HARD_LIQUIDATION_DAY = 29
TURNS_PER_DAY = 24

# These switches document the selected local ablation.  Dynamic replanting and
# JIT seed chasing were rejected because crop lead times made them overreact to
# short-lived spot moves; fast delivery and town-tick timing were retained.
USE_ADAPTIVE_PLANTING = False
USE_JIT_SEEDS = False
USE_EXPRESS_DROP = True
USE_REACTIVE_SELLING = False
USE_FORECAST_GATES = False
USE_TICK_TIMING = True


CROPS = {
    "WHEAT": {
        "seed_cost": 10,
        "base_price": 25,
        "first_yield_day": 2,
        "harvest_day": 4,
        "last_plant_day": 24,
        "yield": 4,
        "duration": 4,
        "ongoing": False,
    },
    "CARROT": {
        "seed_cost": 20,
        "base_price": 35,
        "first_yield_day": 2,
        "harvest_day": 3,
        "last_plant_day": 25,
        "yield": 3,
        "duration": 3,
        "ongoing": False,
    },
    "TOMATO": {
        "seed_cost": 50,
        "base_price": 60,
        "first_yield_day": 8,
        "harvest_day": 8,
        "last_plant_day": 20,
        "yield": 4,
        "duration": 11,
        "ongoing": True,
    },
    "STRAWBERRY": {
        "seed_cost": 100,
        "base_price": 120,
        "first_yield_day": 10,
        "harvest_day": 10,
        "last_plant_day": 18,
        "yield": 4,
        "duration": 16,
        "ongoing": True,
    },
    "MELON": {
        "seed_cost": 80,
        "base_price": 250,
        "first_yield_day": 10,
        # Daily watering reaches the unfertilized six-unit cap at age ten.
        "harvest_day": 10,
        "last_plant_day": 18,
        "yield": 6,
        "duration": 10,
        "ongoing": False,
    },
}


PRODUCTS = tuple(CROPS)
MANAGED_PLOTS = tuple((x, y) for y in range(5) for x in range(5))

# V2's exact layout remains available for controlled ablations.
LEGACY_SLOTS = (
    (0, 0, "MELON", "CARROT"),
    (1, 0, "STRAWBERRY", "WHEAT"),
    (2, 0, "MELON", "CARROT"),
    (3, 0, "CARROT", "WHEAT"),
    (4, 0, "MELON", "CARROT"),
    (0, 1, "WHEAT", "CARROT"),
    (1, 1, "MELON", "CARROT"),
    (2, 1, "STRAWBERRY", "WHEAT"),
    (3, 1, "MELON", "CARROT"),
    (4, 1, "TOMATO", "CARROT"),
    (0, 2, "MELON", "CARROT"),
    (1, 2, "CARROT", "WHEAT"),
    (2, 2, "MELON", "CARROT"),
    (3, 2, "STRAWBERRY", "WHEAT"),
    (4, 2, "MELON", "CARROT"),
    (0, 3, "STRAWBERRY", "WHEAT"),
    (1, 3, "MELON", "CARROT"),
    (2, 3, "WHEAT", "CARROT"),
    (3, 3, "MELON", "CARROT"),
    (4, 3, "TOMATO", "CARROT"),
    (0, 4, "MELON", "CARROT"),
    (1, 4, "CARROT", "WHEAT"),
    (2, 4, "STRAWBERRY", "WHEAT"),
    (3, 4, "WHEAT", "CARROT"),
    (4, 4, "MELON", "CARROT"),
)


MARKET_PARAMS = {
    "WHEAT": {
        "base": 25, "I0": 10000, "T": 400,
        "below_func": "sqrt", "below_target": 0.80,
        "above_func": "log", "above_target": 0.20,
    },
    "CARROT": {
        "base": 35, "I0": 10000, "T": 450,
        "below_func": "hinge", "below_target": 1.00,
        "above_func": "sqrt", "above_target": 0.70,
    },
    "TOMATO": {
        "base": 60, "I0": 10000, "T": 200,
        "below_func": "hinge", "below_target": 0.40,
        "above_func": "sqrt", "above_target": 0.60,
    },
    "STRAWBERRY": {
        "base": 120, "I0": 10000, "T": 100,
        "below_func": "sqrt", "below_target": 0.70,
        "above_func": "linear", "above_target": 1.60,
    },
    "MELON": {
        "base": 250, "I0": 10000, "T": 300,
        "below_func": "log", "below_target": 0.20,
        "above_func": "sq", "above_target": 3.60,
    },
}


SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}


SALE_FLOORS = {
    "WHEAT": 20,
    "CARROT": 24,
    "TOMATO": 44,
    "STRAWBERRY": 92,
    "MELON": 180,
}


SALE_BATCHES = {
    "WHEAT": 18,
    "CARROT": 12,
    "TOMATO": 6,
    "STRAWBERRY": 4,
    "MELON": 6,
}


LEGACY_SELL_RULES = {
    "WHEAT": (16, 18),
    "CARROT": (12, 24),
    "TOMATO": (6, 36),
    "STRAWBERRY": (4, 78),
    "MELON": (6, 160),
}


# Memory is keyed by player because local self-play can share one module.
_MEMORY = {}


def _safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _safe_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def _clip(value, low, high):
    return max(low, min(high, value))


def _step_toward(position, target):
    x, y = position
    tx, ty = target
    dx = tx - x
    dy = ty - y
    if abs(dx) >= abs(dy) and dx:
        return ["EAST" if dx > 0 else "WEST"]
    if dy:
        return ["SOUTH" if dy > 0 else "NORTH"]
    return PASS


def _nearest_shed_tile(position):
    access = ((4, 4), (5, 4), (4, 5), (5, 5))
    x, y = position
    return min(access, key=lambda p: (abs(p[0] - x) + abs(p[1] - y), p[1], p[0]))


def _shape(name, x, scale):
    x = max(0.0, float(x))
    if name == "linear":
        return x
    if name == "sq":
        return x * x
    if name == "sqrt":
        return x ** 0.5
    if name == "log":
        return math.log(1.0 + x)
    if name == "log10":
        return math.log10(1.0 + x)
    if name == "hinge":
        if scale <= 0:
            return x
        ratio = x / scale
        return ratio + 8.0 * max(0.0, ratio - 1.0) ** 2
    return x


def _item_params(item, market):
    params = dict(MARKET_PARAMS[item])
    override = ((market or {}).get("params", {}) or {}).get(item)
    if isinstance(override, dict):
        params.update(override)
    return params


def _market_price(item, inventory, market):
    params = _item_params(item, market)
    base = _safe_float(params.get("base"), CROPS[item]["base_price"])
    anchor = _safe_float(params.get("I0"), 10000)
    scale = max(1.0, _safe_float(params.get("T"), 1))
    if inventory < anchor:
        name = params.get("below_func", "linear")
        target = _safe_float(params.get("below_target"), 0)
        amplitude = target * base / max(1e-9, _shape(name, scale, scale))
        price = base + amplitude * _shape(name, anchor - inventory, scale)
    else:
        name = params.get("above_func", "linear")
        target = _safe_float(params.get("above_target"), 0)
        amplitude = target * base / max(1e-9, _shape(name, scale, scale))
        price = base - amplitude * _shape(name, inventory - anchor, scale)
    return max(1, int(round(price)))


def _batch_revenue(item, inventory, quantity, market, collision_units=0):
    total = 0
    added = 0
    other_left = max(0, _safe_int(collision_units))
    for _ in range(max(0, _safe_int(quantity))):
        price = _market_price(item, inventory, market)
        total += price
        if price > 1:
            inventory += 1
            added += 1
            if other_left > 0:
                inventory += 1
                other_left -= 1
    return total, inventory, added


def _shop_tick_units(town):
    units = {item: 0 for item in PRODUCTS}
    for shop in ((town or {}).get("unlocked_shops", []) or []):
        products = SHOP_PRODUCTS.get(shop, ())
        multiplier = 2 if len(products) == 1 else 1
        for item in products:
            if item in units:
                units[item] += multiplier
    return units


def _daily_demand(town):
    tick = _shop_tick_units(town)
    return {item: 1 + 6 * tick[item] for item in PRODUCTS}


def _town_after_step(step, shops, item):
    town = {"unlocked_shops": shops}
    consumed = _shop_tick_units(town)[item] if step % 4 == 0 else 0
    if step % 24 == 0:
        consumed += 1
    return consumed


def _crop_counts(farm):
    counts = {crop: 0 for crop in CROPS}
    for row in (farm.get("tiles", []) or []):
        for tile in row:
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                if crop in counts:
                    counts[crop] += 1
    return counts


def _crop_pipeline(farm, day):
    """Estimate visible units that can reach the market from existing plants."""
    pipeline = {crop: 0.0 for crop in CROPS}
    ready = {crop: 0.0 for crop in CROPS}
    for row in (farm.get("tiles", []) or []):
        for tile in row:
            if not (isinstance(tile, dict) and tile.get("kind") == "PLANT"):
                continue
            crop = tile.get("crop")
            if crop not in CROPS:
                continue
            data = CROPS[crop]
            age = day - _safe_int(tile.get("planted_day"), day)
            held = max(0, _safe_int(tile.get("yield_units"), 0))
            health = 0.45 if _safe_int(tile.get("consecutive_unwatered"), 0) else 1.0
            if data["ongoing"]:
                produced = max(0, age - data["first_yield_day"] + 1)
                remaining = max(0, data["yield"] - produced)
                estimate = held + remaining
            else:
                estimate = max(held, data["yield"])
            pipeline[crop] += health * estimate
            if age >= data["first_yield_day"] and held > 0:
                ready[crop] += held
    return pipeline, ready


def _market_signals(player, step, market, town, opponent, day):
    inventory = (market or {}).get("inventory", {}) or {}
    prices = (market or {}).get("prices", {}) or {}
    opponent_pipeline, opponent_ready = _crop_pipeline(opponent, day)
    previous = _MEMORY.get(player)
    valid_previous = previous is not None and previous.get("step") == step - 1
    supply_ema = {}
    price_delta = {}
    harvested = {}

    for item in PRODUCTS:
        params = _item_params(item, market)
        scale = max(1.0, _safe_float(params.get("T"), 1))
        current_inventory = _safe_float(inventory.get(item), params.get("I0", 10000))
        current_price = _safe_float(prices.get(item), params.get("base", 1))
        if valid_previous:
            previous_inventory = _safe_float(
                previous.get("inventory", {}).get(item), current_inventory
            )
            known_town = _town_after_step(
                previous["step"], previous.get("shops", []), item
            )
            own_added = _safe_float(previous.get("own_added", {}).get(item), 0)
            other_supply = current_inventory - previous_inventory + known_town - own_added
            other_supply = _clip(other_supply, -0.20 * scale, 0.20 * scale)
            old_ema = _safe_float(previous.get("supply_ema", {}).get(item), 0)
            supply_ema[item] = 0.68 * old_ema + 0.32 * other_supply
            price_delta[item] = current_price - _safe_float(
                previous.get("prices", {}).get(item), current_price
            )
            old_pipeline = _safe_float(
                previous.get("opponent_pipeline", {}).get(item), opponent_pipeline[item]
            )
            harvested[item] = max(0.0, old_pipeline - opponent_pipeline[item])
        else:
            supply_ema[item] = 0.0
            price_delta[item] = 0.0
            harvested[item] = 0.0

    return {
        "inventory": {item: _safe_float(inventory.get(item), 10000) for item in PRODUCTS},
        "prices": {
            item: _safe_int(prices.get(item), CROPS[item]["base_price"])
            for item in PRODUCTS
        },
        "supply_ema": supply_ema,
        "price_delta": price_delta,
        "opponent_pipeline": opponent_pipeline,
        "opponent_ready": opponent_ready,
        "opponent_harvested": harvested,
        "opponent_counts": _crop_counts(opponent),
        "demand_per_day": _daily_demand(town),
        "shop_tick": _shop_tick_units(town),
    }


def _remember(player, step, market, town, signals, own_added):
    _MEMORY[player] = {
        "step": step,
        "inventory": dict(signals["inventory"]),
        "prices": dict(signals["prices"]),
        "shops": list((town or {}).get("unlocked_shops", []) or []),
        "supply_ema": dict(signals["supply_ema"]),
        "own_added": dict(own_added),
        "opponent_pipeline": dict(signals["opponent_pipeline"]),
    }


def _legacy_crop(primary, fallback, day, signals):
    price = signals["prices"][primary]
    opponent_count = signals["opponent_counts"].get(primary, 0)
    if USE_FORECAST_GATES and fallback in ("WHEAT", "CARROT"):
        wheat_score = (4 * signals["prices"]["WHEAT"] - 10) / 4.0
        carrot_score = (3 * signals["prices"]["CARROT"] - 20) / 3.0
        wheat_score += 0.10 * signals["demand_per_day"]["WHEAT"]
        carrot_score += 0.10 * signals["demand_per_day"]["CARROT"]
        if signals["supply_ema"]["CARROT"] > 2:
            carrot_score -= 5
        if signals["supply_ema"]["WHEAT"] > 3:
            wheat_score -= 2
        fallback = "WHEAT" if wheat_score >= carrot_score else "CARROT"
    candidate = primary
    if primary == "MELON":
        fragile = price < 160 or opponent_count >= 8
        if USE_FORECAST_GATES:
            fragile = fragile or (
                signals["inventory"]["MELON"] > 10055
                or signals["supply_ema"]["MELON"] > 2
                or signals["opponent_ready"]["MELON"] >= 18
                or signals["opponent_harvested"]["MELON"] > 0
            )
        if fragile:
            candidate = fallback
    elif primary == "STRAWBERRY":
        support = signals["demand_per_day"]["STRAWBERRY"] > 1
        fragile = price < 82 or (day >= 9 and not support and opponent_count >= 5)
        if USE_FORECAST_GATES:
            fragile = fragile or (
                signals["inventory"]["STRAWBERRY"] > 10022
                or signals["supply_ema"]["STRAWBERRY"] > 1.5
                or (day >= 9 and not support and opponent_count >= 3)
            )
        if fragile:
            candidate = fallback
    elif primary == "TOMATO":
        weak = price < 34
        if USE_FORECAST_GATES:
            weak = weak or (
                price < 40
                and signals["demand_per_day"]["TOMATO"] <= 1
                and signals["supply_ema"]["TOMATO"] > 0
            )
        if weak:
            candidate = fallback
    if day <= CROPS[candidate]["last_plant_day"]:
        return candidate
    if day <= CROPS[fallback]["last_plant_day"]:
        return fallback
    if day <= CROPS["CARROT"]["last_plant_day"]:
        return "CARROT"
    if day <= CROPS["WHEAT"]["last_plant_day"]:
        return "WHEAT"
    return None


def _adaptive_caps(signals, active):
    support = signals["demand_per_day"]
    opponent = signals["opponent_counts"]
    melon_cap = 8
    if (
        opponent.get("MELON", 0) >= 8
        or signals["inventory"]["MELON"] > 10060
        or signals["supply_ema"]["MELON"] > 3
        or signals["prices"]["MELON"] < 165
    ):
        melon_cap = max(active.get("MELON", 0), 2)
    elif opponent.get("MELON", 0) >= 5:
        melon_cap = 5

    strawberry_cap = 3
    if support["STRAWBERRY"] >= 13:
        strawberry_cap = 5
    elif support["STRAWBERRY"] >= 7:
        strawberry_cap = 4
    if signals["prices"]["STRAWBERRY"] < 72 or signals["supply_ema"]["STRAWBERRY"] > 2:
        strawberry_cap = max(active.get("STRAWBERRY", 0), 1)

    return {
        "WHEAT": 25,
        "CARROT": 14,
        "TOMATO": 4,
        "STRAWBERRY": strawberry_cap,
        "MELON": melon_cap,
    }


def _minimum_mix(day, signals):
    minimum = {"WHEAT": 5, "CARROT": 5, "TOMATO": 0, "STRAWBERRY": 0, "MELON": 0}
    if day <= CROPS["TOMATO"]["last_plant_day"] and signals["prices"]["TOMATO"] >= 35:
        minimum["TOMATO"] = 2
    if day <= CROPS["STRAWBERRY"]["last_plant_day"] and signals["prices"]["STRAWBERRY"] >= 78:
        minimum["STRAWBERRY"] = 2
    if (
        day <= CROPS["MELON"]["last_plant_day"]
        and signals["prices"]["MELON"] >= 155
        and signals["opponent_counts"].get("MELON", 0) < 8
        and signals["inventory"]["MELON"] < 10070
    ):
        minimum["MELON"] = 4
    return minimum


def _cohort_caps(day):
    return {
        "WHEAT": 12,
        "CARROT": 9,
        "TOMATO": 2,
        "STRAWBERRY": 2 if day == 0 else 1,
        "MELON": 4 if day == 0 else 2,
    }


def _projected_crop_score(crop, market, signals, own_pipeline, virtual_units, owned_seed):
    data = CROPS[crop]
    horizon = data["first_yield_day"]
    params = _item_params(crop, market)
    scale = max(1.0, _safe_float(params.get("T"), 1))
    momentum = _clip(signals["supply_ema"][crop] * 10.0, -0.15 * scale, 0.15 * scale)
    forecast_inventory = (
        signals["inventory"][crop]
        + own_pipeline[crop]
        + virtual_units[crop]
        + 0.65 * signals["opponent_pipeline"][crop]
        + momentum
        - signals["demand_per_day"][crop] * horizon
    )
    quantity = data["yield"]
    solo_revenue, _, _ = _batch_revenue(crop, forecast_inventory, quantity, market)
    collision = min(
        quantity,
        int(round(signals["opponent_ready"][crop] + 0.25 * signals["opponent_pipeline"][crop])),
    )
    collision_revenue, _, _ = _batch_revenue(
        crop, forecast_inventory, quantity, market, collision
    )
    vulnerable = crop in ("MELON", "STRAWBERRY")
    safe_weight = 0.52 if vulnerable else 0.72
    expected_revenue = safe_weight * solo_revenue + (1.0 - safe_weight) * collision_revenue
    profit_per_day = (expected_revenue - data["seed_cost"]) / max(1, data["duration"])
    if owned_seed:
        profit_per_day += data["seed_cost"] / max(1, data["duration"])
    if signals["price_delta"][crop] > 0:
        profit_per_day += min(5.0, 0.25 * signals["price_delta"][crop])
    if signals["supply_ema"][crop] > 0:
        risk = 1.8 if vulnerable else 0.35
        profit_per_day -= risk * min(10.0, signals["supply_ema"][crop])
    return profit_per_day


def _adaptive_plan(farm, private, day, market, signals):
    tiles = farm.get("tiles", []) or []
    active = _crop_counts(farm)
    cohort = {crop: 0 for crop in CROPS}
    for row in tiles:
        for tile in row:
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                if crop in cohort and _safe_int(tile.get("planted_day"), -1) == day:
                    cohort[crop] += 1

    caps = _adaptive_caps(signals, active)
    minimum = _minimum_mix(day, signals)
    cohort_caps = _cohort_caps(day)
    own_pipeline, _ = _crop_pipeline(farm, day)
    virtual_units = {crop: 0.0 for crop in CROPS}
    seeds = private.get("seeds", {}) or {}
    plan = {}

    for x, y in MANAGED_PLOTS:
        try:
            tile = tiles[y][x]
        except (IndexError, TypeError):
            continue
        empty = tile is None or (isinstance(tile, dict) and tile.get("kind") == "WEED")
        if not empty:
            continue
        candidates = []
        for crop, data in CROPS.items():
            if day > data["last_plant_day"]:
                continue
            if active[crop] >= caps[crop] or cohort[crop] >= cohort_caps[crop]:
                continue
            price = signals["prices"][crop]
            hard_floor = {"WHEAT": 14, "CARROT": 15, "TOMATO": 28, "STRAWBERRY": 58, "MELON": 105}[crop]
            if price < hard_floor and _safe_int(seeds.get(crop), 0) <= 0:
                continue
            score = _projected_crop_score(
                crop,
                market,
                signals,
                own_pipeline,
                virtual_units,
                _safe_int(seeds.get(crop), 0) > 0,
            )
            deficit = max(0, minimum[crop] - active[crop])
            if deficit:
                score += 210.0 + 8.0 * deficit
            candidates.append((score, CROPS[crop]["base_price"], crop))

        if not candidates:
            plan[(x, y)] = None
            continue
        score, _, crop = max(candidates)
        if score <= 0:
            plan[(x, y)] = None
            continue
        plan[(x, y)] = crop
        active[crop] += 1
        cohort[crop] += 1
        virtual_units[crop] += CROPS[crop]["yield"]
    return plan


def _legacy_plan(farm, day, signals):
    tiles = farm.get("tiles", []) or []
    plan = {}
    for x, y, primary, fallback in LEGACY_SLOTS:
        try:
            tile = tiles[y][x]
        except (IndexError, TypeError):
            continue
        if tile is None or (isinstance(tile, dict) and tile.get("kind") == "WEED"):
            plan[(x, y)] = _legacy_crop(primary, fallback, day, signals)
    return plan


def _crop_plan(farm, private, day, market, signals):
    if USE_ADAPTIVE_PLANTING:
        return _adaptive_plan(farm, private, day, market, signals)
    return _legacy_plan(farm, day, signals)


def _tile_task(tile, crop_to_plant, day, hour):
    if tile is None:
        if crop_to_plant is not None and hour <= FINAL_PLANT_HOUR:
            return 3, ["PLANT", crop_to_plant]
        return None
    if tile == "LOCKED" or not isinstance(tile, dict):
        return None
    kind = tile.get("kind")
    if kind == "WEED":
        return 2, ["DIG"]
    if kind != "PLANT":
        return None
    crop = tile.get("crop")
    data = CROPS.get(crop)
    if data is None:
        return 2, ["DIG"]
    age = day - _safe_int(tile.get("planted_day"), day)
    held = max(0, _safe_int(tile.get("yield_units"), 0))
    watered = bool(tile.get("watered_today", False))
    missed = _safe_int(tile.get("consecutive_unwatered"), 0)

    if day >= FINAL_FARM_DAY and held > 0 and age >= data["first_yield_day"]:
        return 0, ["HARVEST"]
    if data["ongoing"]:
        if held > 0 and age >= data["first_yield_day"]:
            return 0, ["HARVEST"]
        if not watered:
            return 1, ["WATER"]
        return None
    if held > 0 and age >= data["harvest_day"]:
        if age == data["harvest_day"] and not watered:
            return 0, ["WATER"]
        return 0, ["HARVEST"]
    if not watered:
        return 1, ["WATER"]
    return None


def _total_exposure(private):
    total = 0
    for value in (private.get("shed", {}) or {}).values():
        total += max(0, _safe_int(value))
    for inventory in (private.get("inventories", []) or []):
        for value in (inventory or {}).values():
            total += max(0, _safe_int(value))
    return total


def _should_express(inventory, signals, day, exposure):
    if not USE_EXPRESS_DROP or not inventory:
        return False
    if day >= SOFT_LIQUIDATION_DAY or exposure >= SOFT_EXPOSURE:
        return True
    competing_premium = any(
        signals["opponent_counts"].get(item, 0) > 0
        for item in ("MELON", "STRAWBERRY", "TOMATO")
    )
    if (
        competing_premium
        and sum(max(0, _safe_int(v)) for v in inventory.values()) >= 7
    ):
        return True
    for item, quantity in inventory.items():
        if item not in CROPS or _safe_int(quantity) <= 0:
            continue
        danger = (
            signals["supply_ema"][item] > 0.75
            or signals["opponent_ready"][item] > 0
            or signals["opponent_harvested"][item] > 0
        )
        if item in ("MELON", "STRAWBERRY", "TOMATO"):
            competing = signals["opponent_counts"].get(item, 0) > 0
            if danger or (
                competing
                and signals["prices"][item] >= int(0.82 * SALE_FLOORS[item])
            ):
                return True
    return False


def _build_tasks(farm, private, plan, day, hour):
    seeds_available = {
        crop: max(0, _safe_int((private.get("seeds", {}) or {}).get(crop), 0))
        for crop in CROPS
    }
    tiles = farm.get("tiles", []) or []
    tasks = []
    for x, y in MANAGED_PLOTS:
        try:
            tile = tiles[y][x]
        except (IndexError, TypeError):
            continue
        task = _tile_task(tile, plan.get((x, y)), day, hour)
        if task is None:
            continue
        priority, action = task
        if action[0] == "PLANT":
            crop = action[1]
            if seeds_available[crop] <= 0:
                continue
            seeds_available[crop] -= 1
        tasks.append({"priority": priority, "target": (x, y), "action": action})
    return tasks


def _assign_unit_actions(farm, private, plan, day, hour, signals):
    positions = [tuple(farm.get("farmer", (4, 4)))]
    positions.extend(tuple(position) for position in (farm.get("hands", []) or []))
    inventories = list(private.get("inventories", []) or [])
    while len(inventories) < len(positions):
        inventories.append({})
    actions = [PASS for _ in positions]
    predicted_drop = {crop: 0 for crop in CROPS}
    remaining_units = set(range(len(positions)))
    exposure = _total_exposure(private)

    express_candidates = []
    for unit_index, position in enumerate(positions):
        inventory = inventories[unit_index] or {}
        if not _should_express(inventory, signals, day, exposure):
            continue
        target = _nearest_shed_tile(position)
        distance = abs(position[0] - target[0]) + abs(position[1] - target[1])
        market_value = sum(
            max(0, _safe_int(quantity)) * signals["prices"].get(item, 0)
            for item, quantity in inventory.items()
            if item in CROPS
        )
        express_candidates.append(
            (market_value / max(1, distance), -distance, -unit_index, unit_index, target)
        )

    # One courier is enough to front-run a vulnerable batch without abandoning
    # the whole field.  Capacity pressure and liquidation may release all units.
    express_candidates.sort(reverse=True)
    express_limit = len(express_candidates) if (
        exposure >= HARD_EXPOSURE or day >= SOFT_LIQUIDATION_DAY
    ) else 1
    for _, _, _, unit_index, target in express_candidates[:express_limit]:
        position = positions[unit_index]
        inventory = inventories[unit_index] or {}
        if position == target:
            actions[unit_index] = ["DROP"]
            for item, quantity in inventory.items():
                if item in predicted_drop:
                    predicted_drop[item] += max(0, _safe_int(quantity))
        else:
            actions[unit_index] = _step_toward(position, target)
        remaining_units.discard(unit_index)

    tasks = _build_tasks(farm, private, plan, day, hour)
    while tasks and remaining_units:
        choices = []
        for unit_index in remaining_units:
            ux, uy = positions[unit_index]
            for task_index, task in enumerate(tasks):
                tx, ty = task["target"]
                distance = abs(tx - ux) + abs(ty - uy)
                choices.append(
                    (
                        task["priority"],
                        distance,
                        ty,
                        tx,
                        unit_index,
                        task_index,
                    )
                )
        if not choices:
            break
        _, _, _, _, unit_index, task_index = min(choices)
        task = tasks.pop(task_index)
        remaining_units.remove(unit_index)
        if positions[unit_index] == task["target"]:
            actions[unit_index] = task["action"]
        else:
            actions[unit_index] = _step_toward(positions[unit_index], task["target"])
    return actions, predicted_drop


def _available_to_sell(private, predicted_drop):
    available = {
        item: max(0, _safe_int((private.get("shed", {}) or {}).get(item), 0))
        for item in CROPS
    }
    for item, quantity in predicted_drop.items():
        available[item] += max(0, _safe_int(quantity))
    return available


def _safe_quantity(item, held, market, signals, floor, cap, collision):
    inventory = signals["inventory"][item]
    quantity = 0
    other_left = max(0, int(round(collision)))
    for _ in range(min(held, cap)):
        price = _market_price(item, inventory, market)
        if price < floor:
            break
        quantity += 1
        if price > 1:
            inventory += 1
            if other_left > 0:
                inventory += 1
                other_left -= 1
    return quantity


def _reactive_sell_orders(private, predicted_drop, market, town, day, step, signals):
    available = _available_to_sell(private, predicted_drop)
    exposure = _total_exposure(private) + sum(predicted_drop.values())
    hard_pressure = exposure >= HARD_EXPOSURE
    soft_pressure = exposure >= SOFT_EXPOSURE
    candidates = []

    for item in ("MELON", "STRAWBERRY", "TOMATO", "CARROT", "WHEAT"):
        held = available[item]
        if held <= 0:
            continue
        price = signals["prices"][item]
        base = CROPS[item]["base_price"]
        danger = (
            signals["supply_ema"][item] > 0.75
            or signals["price_delta"][item] < 0
            or signals["opponent_ready"][item] > 0
            or signals["opponent_harvested"][item] > 0
        )
        upcoming_demand = 0
        if step % 4 == 0:
            upcoming_demand += signals["shop_tick"][item]
        if step % 24 == 0:
            upcoming_demand += 1
        terminal = day >= HARD_LIQUIDATION_DAY
        soft_terminal = day >= SOFT_LIQUIDATION_DAY

        # Town demand is applied after this turn's market queue.  Wait one turn
        # for the better quote unless supply risk, capacity, or season end wins.
        if upcoming_demand and not (danger or soft_pressure or soft_terminal):
            continue

        floor = SALE_FLOORS[item]
        if danger:
            floor = max(1, int(0.78 * floor))
        if soft_pressure or soft_terminal:
            floor = max(1, int(0.58 * floor))
        if hard_pressure or terminal:
            floor = 1

        trigger = (
            terminal
            or hard_pressure
            or soft_terminal
            or soft_pressure
            or price >= SALE_FLOORS[item]
            or (danger and price >= int(0.52 * base))
        )
        if not trigger:
            continue

        cap = SALE_BATCHES[item]
        if soft_terminal:
            cap *= 2
        if terminal:
            cap = held
        collision = 0
        if danger:
            collision = min(
                held,
                signals["opponent_ready"][item]
                + 0.5 * signals["opponent_harvested"][item],
            )
        quantity = _safe_quantity(item, held, market, signals, floor, cap, collision)
        if quantity <= 0 and (hard_pressure or terminal):
            quantity = min(held, cap)
        if quantity <= 0:
            continue
        vulnerability = {"MELON": 5, "STRAWBERRY": 4, "TOMATO": 3, "CARROT": 2, "WHEAT": 1}[item]
        urgency = (
            1000 * int(terminal)
            + 300 * int(hard_pressure)
            + 80 * int(danger)
            + 20 * vulnerability
            + price / max(1, base)
        )
        candidates.append((urgency, item, quantity))

    candidates.sort(reverse=True)
    return [["SELL", item, quantity] for _, item, quantity in candidates]


def _legacy_sell_orders(private, predicted_drop, market, day, step, signals):
    shed = private.get("shed", {}) or {}
    available = {
        item: max(0, _safe_int(shed.get(item), 0))
        for item in CROPS
    }
    if USE_TICK_TIMING:
        for item, quantity in predicted_drop.items():
            if item in available:
                available[item] += max(0, _safe_int(quantity))
    total = (
        sum(available.values())
        if USE_TICK_TIMING
        else sum(max(0, _safe_int(value)) for value in shed.values())
    )
    orders = []
    for item in ("MELON", "STRAWBERRY", "TOMATO", "CARROT", "WHEAT"):
        held = available[item]
        if held <= 0:
            continue
        batch, threshold = LEGACY_SELL_RULES[item]
        danger = (
            signals["supply_ema"][item] > 0.75
            or signals["price_delta"][item] < 0
            or signals["opponent_ready"][item] > 0
            or signals["opponent_harvested"][item] > 0
        )
        # A shop tick can remove several units and materially improve the next
        # quote.  The town center removes only one unit per day, so delaying
        # hour-zero cash for that tiny move costs more than it earns.
        upcoming_demand = step % 4 == 0 and signals["shop_tick"][item] > 0
        if (
            USE_TICK_TIMING
            and upcoming_demand
            and day < 25
            and total < SOFT_EXPOSURE
            and not danger
        ):
            continue
        if day >= 25 or total >= 55 or signals["prices"][item] >= threshold:
            quantity = held if day >= 25 else min(held, batch)
            orders.append(["SELL", item, quantity])
    return orders


def _seed_orders(farm, private, plan, day, hour, market_slots):
    if market_slots <= 0 or day >= FINAL_FARM_DAY or hour > FINAL_PLANT_HOUR:
        return []
    wanted = {crop: 0 for crop in CROPS}
    for crop in plan.values():
        if crop in wanted:
            wanted[crop] += 1
    seeds = private.get("seeds", {}) or {}
    missing = {
        crop: max(0, wanted[crop] - _safe_int(seeds.get(crop), 0))
        for crop in CROPS
    }
    cash = _safe_float(farm.get("money"), 0)
    reserve = 250.0

    if not USE_JIT_SEEDS:
        orders = []
        for crop in ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"):
            if len(orders) >= market_slots or missing[crop] <= 0:
                continue
            cost = CROPS[crop]["seed_cost"]
            quantity = min(missing[crop], max(0, int((cash - reserve) // cost)))
            if quantity > 0:
                orders.append(["BUY_SEED", crop, quantity])
                cash -= quantity * cost
        return orders

    per_turn_cap = {"WHEAT": 3, "CARROT": 3, "TOMATO": 2, "STRAWBERRY": 1, "MELON": 2}
    allocation = {crop: 0 for crop in CROPS}
    order = ("MELON", "STRAWBERRY", "TOMATO", "CARROT", "WHEAT")
    total_limit = min(6, max(1, 2 * (1 + len(farm.get("hands", []) or []))))
    changed = True
    while sum(allocation.values()) < total_limit and changed:
        changed = False
        for crop in order:
            if sum(allocation.values()) >= total_limit:
                break
            if allocation[crop] >= min(missing[crop], per_turn_cap[crop]):
                continue
            cost = CROPS[crop]["seed_cost"]
            if cash - cost < reserve:
                continue
            allocation[crop] += 1
            cash -= cost
            changed = True

    orders = []
    for crop in order:
        if allocation[crop] > 0 and len(orders) < market_slots:
            orders.append(["BUY_SEED", crop, allocation[crop]])
    return orders


def _market_orders(
    farm, private, predicted_drop, plan, market, town, day, hour, step, signals
):
    if USE_REACTIVE_SELLING:
        orders = _reactive_sell_orders(
            private, predicted_drop, market, town, day, step, signals
        )
    else:
        orders = _legacy_sell_orders(
            private, predicted_drop, market, day, step, signals
        )

    if hour == 0 and day <= FINAL_FARM_DAY:
        current_hands = len(farm.get("hands", []) or [])
        for _ in range(max(0, DESIRED_HANDS - current_hands)):
            if len(orders) >= MAX_MARKET_ORDERS:
                break
            orders.append(["HIRE"])

    free_slots = MAX_MARKET_ORDERS - len(orders)
    orders.extend(_seed_orders(farm, private, plan, day, hour, free_slots))
    orders = orders[:MAX_MARKET_ORDERS]

    own_added = {crop: 0 for crop in CROPS}
    for order in orders:
        if len(order) >= 3 and order[0] == "SELL" and order[1] in own_added:
            item = order[1]
            _, _, added = _batch_revenue(
                item, signals["inventory"][item], order[2], market
            )
            own_added[item] += added
    return orders, own_added


def agent(obs):
    """Required Kaggle entrypoint."""
    try:
        farms = obs.get("farms", []) or []
        player = _safe_int(obs.get("player"), 0)
        private = obs.get("private", {}) or {}
        if player < 0 or player >= len(farms):
            return {"farmer": PASS, "hands": [], "market": []}

        farm = farms[player]
        opponent = farms[1 - player] if len(farms) == 2 else {}
        day = _safe_int(obs.get("day"), 0)
        hour = _safe_int(obs.get("hour"), 0)
        step = day * TURNS_PER_DAY + hour
        market = obs.get("market", {}) or {}
        town = obs.get("town", {}) or {}
        signals = _market_signals(player, step, market, town, opponent, day)
        plan = _crop_plan(farm, private, day, market, signals)

        if day > FINAL_FARM_DAY:
            unit_actions = [PASS]
            unit_actions.extend(PASS for _ in (farm.get("hands", []) or []))
            predicted_drop = {crop: 0 for crop in CROPS}
        else:
            unit_actions, predicted_drop = _assign_unit_actions(
                farm, private, plan, day, hour, signals
            )

        market_orders, own_added = _market_orders(
            farm,
            private,
            predicted_drop,
            plan,
            market,
            town,
            day,
            hour,
            step,
            signals,
        )
        _remember(player, step, market, town, signals, own_added)

        farmer_action = unit_actions[0] if unit_actions else PASS
        hand_actions = unit_actions[1:]
        return {
            "farmer": farmer_action,
            "hands": hand_actions,
            "market": market_orders,
        }
    except Exception:
        hands = []
        try:
            farms = obs.get("farms", []) or []
            player = _safe_int(obs.get("player"), 0)
            if 0 <= player < len(farms):
                hands = [PASS for _ in (farms[player].get("hands", []) or [])]
        except Exception:
            hands = []
        return {"farmer": PASS, "hands": hands, "market": []}

## 3. Check the entrypoint and action contract

This catches the filename/function mismatch that the tutorial's in-memory callable test misses.

In [ ]:
import ast
import importlib.util
import json
from pathlib import Path

main_path = Path("main.py")
assert main_path.is_file(), "main.py was not created"

tree = ast.parse(main_path.read_text(encoding="utf-8"), filename="main.py")
function_names = {node.name for node in tree.body if isinstance(node, ast.FunctionDef)}
assert "agent" in function_names, "main.py must expose def agent(obs)"

spec = importlib.util.spec_from_file_location("submission_agent", main_path)
submission_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_agent)
assert callable(submission_agent.agent)

tiles = [
    [None if x < 5 and y < 5 else "LOCKED" for x in range(10)]
    for y in range(10)
]
dummy_obs = {
    "player": 0,
    "day": 0,
    "hour": 0,
    "farms": [{
        "money": 3000,
        "tiles": tiles,
        "farmer": [4, 4],
        "hands": [],
        "unlocked_quadrants": ["NW"],
        "hires_today": 0,
    }],
    "private": {"shed": {}, "seeds": {}, "inventories": [{}]},
    "market": {"inventory": {}, "prices": {}},
    "town": {"unlocked_shops": []},
}

action = submission_agent.agent(dummy_obs)
assert set(action) == {"farmer", "hands", "market"}
assert isinstance(action["farmer"], list) and action["farmer"]
assert isinstance(action["hands"], list)
assert isinstance(action["market"], list) and len(action["market"]) <= 10
json.dumps(action)
print("Entrypoint and JSON action contract: OK")
print(action)

## 4. Run full file-loader validation games

These tests execute the exact `main.py` filepath for all 720 turns. Self-play
mirrors Kaggle's Validation Episode; `starter` and `random` provide independent
smoke tests. Rewards validate execution, not future leaderboard performance.

In [ ]:
from kaggle_environments import make

opponents = ["main.py", "starter", "random"]
for index, opponent in enumerate(opponents):
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": 20261001 + index},
        debug=True,
    )
    env.run(["main.py", opponent])
    final = env.steps[-1]
    statuses = [state.status for state in final]
    rewards = [state.reward for state in final]
    assert statuses == ["DONE", "DONE"], (opponent, statuses)
    print(f"vs {opponent:8s}: statuses={statuses}, rewards={rewards}")

print("V4 full 720-turn file-loader validation: OK")

## 5. Build and verify the submission archive

The member name must be exactly `main.py`, with no enclosing directory.

In [ ]:
import hashlib
import tarfile
from pathlib import Path

archive_path = Path("submission.tar.gz")
with tarfile.open(archive_path, "w:gz") as archive:
    archive.add("main.py", arcname="main.py", recursive=False)

with tarfile.open(archive_path, "r:gz") as archive:
    members = archive.getnames()
    assert members == ["main.py"], members
    archived_source = archive.extractfile("main.py").read()

assert archived_source == Path("main.py").read_bytes()
size_mib = archive_path.stat().st_size / (1024 * 1024)
assert size_mib < 100, f"Archive is too large: {size_mib:.2f} MiB"
sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()

print(f"Created: {archive_path.resolve()}")
print(f"Members: {members}")
print(f"Size: {size_mib:.4f} MiB")
print(f"SHA-256: {sha256}")

## 6. Submit v4

1. Attach the **Kaggriculture** competition and accept its rules.
2. Run every cell and save a successful notebook version.
3. Confirm validation finishes with `DONE/DONE` and the archive cell reports
   `Members: ['main.py']`.
4. Click **Submit to competition** and select this version's
   `submission.tar.gz` output.